In [0]:
%pip install statsmodels optuna

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose


# Lectura de Datos y manipulacion a pandas

In [0]:
serie_mensual_df= spark.read.table("joel.default.parcial_1_serie_mensual")
serie_mensual_df=serie_mensual_df.toPandas()

In [0]:
# Definiendo la columna mes como indice de tiempo para el modelo
serie_mensual_df['mes'] = pd.to_datetime(serie_mensual_df['mes'])
serie_mensual_df = serie_mensual_df.set_index('mes')
serie_mensual_df


# Análisis exploratorio de datos

## Analisis a nivel de año

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))

serie_mensual_df_per_year = serie_mensual_df.groupby(serie_mensual_df.index.year).sum()
serie_mensual_df_per_year['variacion_%'] = serie_mensual_df_per_year['ventas'].pct_change() * 100

ax.bar(serie_mensual_df_per_year.index, serie_mensual_df_per_year["ventas"], label='Ventas')
ax2 = ax.twinx()
ax2.plot(serie_mensual_df_per_year.index, serie_mensual_df_per_year['variacion_%'], color='red', marker='o', label='Variación % YoY')

ax.set_title('Comparación de ventas por año')
ax.set_xlabel('Año')
ax.set_ylabel('Ventas')
ax2.set_ylabel('Variación % YoY')

ax.legend(loc='upper left')
ax2.legend(loc='upper left', bbox_to_anchor=(0, 0.85))

plt.tight_layout()
plt.show()

Se observa un fuerte crecimiento en las ventas durante los primeros seis años, seguido de un estancamiento o leve disminución entre 1965 y 1967.

## Analisis a nivel de mes

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(serie_mensual_df.index, serie_mensual_df['ventas'], marker='o', linewidth=1.5)

ax.set_title('Serie de tiempo mensual')
ax.set_xlabel('mes')
ax.set_ylabel('ventas')
fig.autofmt_xdate() 
plt.tight_layout()
plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))

for year, group in serie_mensual_df.groupby(serie_mensual_df.index.year):
    ax.plot(group.index.month, group['ventas'], marker='o', label=str(year), linewidth=1.5)

ax.set_title('Comparación de estacionalidad por año')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas')
ax.legend(title='Año')
plt.tight_layout()
plt.show()

Utilizando un metodo grafico podemos observar una estacionalidad marcada, las ventas alcanzan su punto mas alto entre el cuarto y quinto mes del año


# Analisis estadistico de la serie de tiempo

In [0]:

resultado_seasonal_decompose = seasonal_decompose(serie_mensual_df['ventas'], model='additive', period=12)
tendencia_seasonal_decompose = resultado_seasonal_decompose.trend
estacional_seasonal_decompose = resultado_seasonal_decompose.seasonal
residuo_seasonal_decompose = resultado_seasonal_decompose.resid  # ruido + ciclo mezclados

resultado_seasonal_decompose.plot()
plt.tight_layout()
plt.show()

In [0]:
from statsmodels.tsa.seasonal import STL

stl = STL(serie_mensual_df['ventas'], period=12, robust=True)
resultado_stl = stl.fit()
tendencia_stl = resultado_stl.trend
estacional_stl = resultado_stl.seasonal
residuo_stl = resultado_stl.resid
resultado_stl.plot()
plt.tight_layout()
plt.show

EL modelo seasonal decompose nos indica:
- Tendencia: La tendencia crece y presenta un estacamiento en los ultimos años de datos
- Estacionalidad: muestra un patron recurrente y repetivio, lo que indica una fuete estacionalidad
- Residuo: La mayoria de los datos se encuentra cerca de 0, indicando que se se logro extraer la informacion útil 

In [0]:
# Separar los datos en entrenamiento y prueba
train_df = serie_mensual_df[serie_mensual_df.index.year <= 1966]
test_df = serie_mensual_df[serie_mensual_df.index.year == 1967]

print(f"Datos de entrenamiento: {len(train_df)} meses (1960-1966)")
print(f"Datos de prueba: {len(test_df)} meses (1967)")
print(f"\nÚltimo mes de entrenamiento: {train_df.index[-1]}")
print(f"Primer mes de prueba: {test_df.index[0]}")

In [0]:
import optuna
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configurar el experimento de MLflow
mlflow.set_experiment("/Users/joel@ramirezai.com/prueba_de_series_de_tiempo")

def objective(trial):
    """Función objetivo para Optuna que optimiza hiperparámetros de SARIMA"""
    
    # Sugerir hiperparámetros para SARIMA (p,d,q)(P,D,Q,s)
    p = trial.suggest_int('p', 0, 3)
    d = trial.suggest_int('d', 0, 2)
    q = trial.suggest_int('q', 0, 3)
    P = trial.suggest_int('P', 0, 2)
    D = trial.suggest_int('D', 0, 1)
    Q = trial.suggest_int('Q', 0, 2)
    s = 12  # Estacionalidad mensual
    
    try:
        with mlflow.start_run(nested=True):
            # Registrar hiperparámetros
            mlflow.log_params({
                'p': p, 'd': d, 'q': q,
                'P': P, 'D': D, 'Q': Q, 's': s
            })
            
            # Entrenar el modelo SARIMA
            model = SARIMAX(
                train_df['ventas'],
                order=(p, d, q),
                seasonal_order=(P, D, Q, s),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            fitted_model = model.fit(disp=False, maxiter=200)
            
            # Hacer predicciones en el conjunto de prueba
            forecast = fitted_model.forecast(steps=len(test_df))
            
            # Calcular métricas
            rmse = np.sqrt(mean_squared_error(test_df['ventas'], forecast))
            mae = mean_absolute_error(test_df['ventas'], forecast)
            
            # Registrar métricas
            mlflow.log_metrics({
                'rmse': rmse,
                'mae': mae,
                'aic': fitted_model.aic,
                'bic': fitted_model.bic
            })
            
            return rmse  # Optuna minimizará el RMSE
            
    except Exception as e:
        # Si el modelo falla, retornar un valor muy alto
        return float('inf')

print("Iniciando optimización de hiperparámetros con Optuna...")
print("Este proceso puede tomar varios minutos.\n")

In [0]:
# Crear y ejecutar el estudio de Optuna
with mlflow.start_run(run_name="SARIMA_Optuna_Optimization"):
    study = optuna.create_study(direction='minimize', study_name='sarima_optimization')
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    # Registrar los mejores hiperparámetros
    best_params = study.best_params
    mlflow.log_params({
        f'best_{k}': v for k, v in best_params.items()
    })
    mlflow.log_metric('best_rmse', study.best_value)
    
    print("\n" + "="*60)
    print("OPTIMIZACIÓN COMPLETADA")
    print("="*60)
    print(f"\nMejores hiperparámetros encontrados:")
    print(f"  SARIMA({best_params['p']},{best_params['d']},{best_params['q']})" +
          f"({best_params['P']},{best_params['D']},{best_params['Q']},12)")
    print(f"\nMejor RMSE: {study.best_value:.2f}")

In [0]:
# Entrenar el modelo final con los mejores hiperparámetros
best_p = best_params['p']
best_d = best_params['d']
best_q = best_params['q']
best_P = best_params['P']
best_D = best_params['D']
best_Q = best_params['Q']

print("\nEntrenando modelo final...")
final_model = SARIMAX(
    train_df['ventas'],
    order=(best_p, best_d, best_q),
    seasonal_order=(best_P, best_D, best_Q, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
final_fitted = final_model.fit(disp=False)

# Realizar pronóstico para 1967
forecast_1967 = final_fitted.forecast(steps=len(test_df))

# Calcular métricas finales
final_rmse = np.sqrt(mean_squared_error(test_df['ventas'], forecast_1967))
final_mae = mean_absolute_error(test_df['ventas'], forecast_1967)

print("\n" + "="*60)
print("MÉTRICAS EN CONJUNTO DE PRUEBA (1967)")
print("="*60)
print(f"RMSE: {final_rmse:.2f}")
print(f"MAE:  {final_mae:.2f}")
print(f"AIC:  {final_fitted.aic:.2f}")
print(f"BIC:  {final_fitted.bic:.2f}")

In [0]:
# Crear visualización de observado vs pronosticado
fig, ax = plt.subplots(figsize=(14, 6))

# Graficar datos de entrenamiento
ax.plot(train_df.index, train_df['ventas'], 
        label='Entrenamiento (1960-1966)', 
        color='blue', linewidth=2, alpha=0.7)

# Graficar datos observados de 1967
ax.plot(test_df.index, test_df['ventas'], 
        label='Observado (1967)', 
        color='green', linewidth=2, marker='o', markersize=6)

# Graficar pronóstico
ax.plot(test_df.index, forecast_1967, 
        label='Pronosticado (1967)', 
        color='red', linewidth=2, linestyle='--', marker='s', markersize=6)

# Agregar banda de confianza visual
residuals = test_df['ventas'].values - forecast_1967.values
std_residual = np.std(residuals)
ax.fill_between(test_df.index, 
                 forecast_1967 - 1.96*std_residual, 
                 forecast_1967 + 1.96*std_residual,
                 alpha=0.2, color='red', label='Intervalo de confianza 95%')

ax.set_title(f'Pronóstico SARIMA({best_p},{best_d},{best_q})({best_P},{best_D},{best_Q},12) - Observado vs Pronosticado',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Ventas', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

# Agregar texto con métricas
textstr = f'RMSE: {final_rmse:.2f}\nMAE: {final_mae:.2f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

In [0]:
import pickle
import json
from io import BytesIO

# Iniciar un nuevo run para registrar artifacts de SARIMA
with mlflow.start_run(run_name="SARIMA_Final_Model_with_Artifacts"):
    
    # 1. Registrar parámetros del modelo
    mlflow.log_params({
        'model_type': 'SARIMA',
        'p': best_p,
        'd': best_d,
        'q': best_q,
        'P': best_P,
        'D': best_D,
        'Q': best_Q,
        's': 12
    })
    
    # 2. Registrar métricas
    mlflow.log_metrics({
        'rmse': final_rmse,
        'mae': final_mae,
        'aic': final_fitted.aic,
        'bic': final_fitted.bic
    })
    
    # 3. Guardar y registrar el modelo como pickle
    model_filename = "sarima_model.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(final_fitted, f)
    mlflow.log_artifact(model_filename, artifact_path="models")
    
    # 4. Guardar y registrar predicciones
    predictions_df = pd.DataFrame({
        'fecha': test_df.index,
        'observado': test_df['ventas'].values,
        'predicho': forecast_1967.values,
        'error': test_df['ventas'].values - forecast_1967.values
    })
    predictions_filename = "sarima_predictions.csv"
    predictions_df.to_csv(predictions_filename, index=False)
    mlflow.log_artifact(predictions_filename, artifact_path="predictions")
    
    # 5. Guardar y registrar gráfico principal
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(train_df.index, train_df['ventas'], label='Entrenamiento (1960-1966)', 
            color='blue', linewidth=2, alpha=0.7)
    ax.plot(test_df.index, test_df['ventas'], label='Observado (1967)', 
            color='green', linewidth=2, marker='o', markersize=6)
    ax.plot(test_df.index, forecast_1967, label='Pronosticado (1967)', 
            color='red', linewidth=2, linestyle='--', marker='s', markersize=6)
    residuals = test_df['ventas'].values - forecast_1967.values
    std_residual = np.std(residuals)
    ax.fill_between(test_df.index, forecast_1967 - 1.96*std_residual, 
                     forecast_1967 + 1.96*std_residual, alpha=0.2, color='red', 
                     label='Intervalo de confianza 95%')
    ax.set_title(f'SARIMA({best_p},{best_d},{best_q})({best_P},{best_D},{best_Q},12) - Observado vs Pronosticado',
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Fecha', fontsize=12)
    ax.set_ylabel('Ventas', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    
    plot_filename = "sarima_forecast_plot.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(plot_filename, artifact_path="plots")
    plt.close()
    
    # 6. Guardar gráfico de residuales
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Residuales vs tiempo
    axes[0, 0].plot(test_df.index, residuals, marker='o')
    axes[0, 0].axhline(y=0, color='r', linestyle='--')
    axes[0, 0].set_title('Residuales vs Tiempo')
    axes[0, 0].set_xlabel('Fecha')
    axes[0, 0].set_ylabel('Residuales')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Histograma de residuales
    axes[0, 1].hist(residuals, bins=10, edgecolor='black')
    axes[0, 1].set_title('Distribución de Residuales')
    axes[0, 1].set_xlabel('Residual')
    axes[0, 1].set_ylabel('Frecuencia')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title('Q-Q Plot')
    axes[1, 0].grid(True, alpha=0.3)
    
    # ACF de residuales
    from statsmodels.graphics.tsaplots import plot_acf
    plot_acf(residuals, lags=10, ax=axes[1, 1])
    axes[1, 1].set_title('ACF de Residuales')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    residuals_filename = "sarima_residuals_analysis.png"
    plt.savefig(residuals_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(residuals_filename, artifact_path="plots")
    plt.close()
    
    # 7. Guardar resumen del modelo como JSON
    model_summary = {
        'model_type': 'SARIMA',
        'order': (best_p, best_d, best_q),
        'seasonal_order': (best_P, best_D, best_Q, 12),
        'metrics': {
            'rmse': float(final_rmse),
            'mae': float(final_mae),
            'aic': float(final_fitted.aic),
            'bic': float(final_fitted.bic)
        },
        'training_period': '1960-1966',
        'test_period': '1967',
        'n_train': len(train_df),
        'n_test': len(test_df)
    }
    
    summary_filename = "sarima_model_summary.json"
    with open(summary_filename, 'w') as f:
        json.dump(model_summary, f, indent=2)
    mlflow.log_artifact(summary_filename, artifact_path="metadata")
    
    print("✓ Artifacts de SARIMA registrados exitosamente en MLflow:")
    print("  - Modelo (pickle)")
    print("  - Predicciones (CSV)")
    print("  - Gráfico de pronóstico")
    print("  - Análisis de residuales")
    print("  - Resumen del modelo (JSON)")

In [0]:
import pickle
import json

# Iniciar un nuevo run para registrar artifacts de Prophet
with mlflow.start_run(run_name="Prophet_Final_Model_with_Artifacts"):
    
    # 1. Registrar parámetros del modelo
    mlflow.log_params({
        'model_type': 'Prophet',
        'changepoint_prior_scale': best_params_prophet['changepoint_prior_scale'],
        'seasonality_prior_scale': best_params_prophet['seasonality_prior_scale'],
        'holidays_prior_scale': best_params_prophet['holidays_prior_scale'],
        'seasonality_mode': best_params_prophet['seasonality_mode'],
        'changepoint_range': best_params_prophet['changepoint_range'],
        'yearly_seasonality': True,
        'weekly_seasonality': False,
        'daily_seasonality': False
    })
    
    # 2. Registrar métricas
    mlflow.log_metrics({
        'rmse': prophet_rmse,
        'mae': prophet_mae
    })
    
    # 3. Guardar y registrar el modelo como pickle
    model_filename = "prophet_model.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(model_prophet, f)
    mlflow.log_artifact(model_filename, artifact_path="models")
    
    # 4. Guardar y registrar predicciones
    predictions_df = pd.DataFrame({
        'fecha': test_prophet['ds'],
        'observado': test_prophet['y'].values,
        'predicho': forecast_test['yhat'].values,
        'yhat_lower': forecast_test['yhat_lower'].values,
        'yhat_upper': forecast_test['yhat_upper'].values,
        'error': test_prophet['y'].values - forecast_test['yhat'].values
    })
    predictions_filename = "prophet_predictions.csv"
    predictions_df.to_csv(predictions_filename, index=False)
    mlflow.log_artifact(predictions_filename, artifact_path="predictions")
    
    # 5. Guardar y registrar gráfico principal
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(train_prophet['ds'], train_prophet['y'], label='Entrenamiento (1960-1966)', 
            color='blue', linewidth=2, alpha=0.7)
    ax.plot(test_prophet['ds'], test_prophet['y'], label='Observado (1967)', 
            color='green', linewidth=2, marker='o', markersize=6)
    ax.plot(forecast_test['ds'], forecast_test['yhat'], label='Pronosticado Prophet (1967)', 
            color='red', linewidth=2, linestyle='--', marker='s', markersize=6)
    ax.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], 
                     forecast_test['yhat_upper'], alpha=0.2, color='red', 
                     label='Intervalo de confianza 80%')
    ax.set_title('Prophet Optimizado - Observado vs Pronosticado',
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Fecha', fontsize=12)
    ax.set_ylabel('Ventas', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    
    plot_filename = "prophet_forecast_plot.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(plot_filename, artifact_path="plots")
    plt.close()
    
    # 6. Guardar gráfico de componentes de Prophet
    fig = model_prophet.plot_components(forecast)
    components_filename = "prophet_components.png"
    plt.savefig(components_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(components_filename, artifact_path="plots")
    plt.close()
    
    # 7. Guardar análisis de residuales
    residuals_prophet = test_prophet['y'].values - forecast_test['yhat'].values
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Residuales vs tiempo
    axes[0, 0].plot(test_prophet['ds'], residuals_prophet, marker='o')
    axes[0, 0].axhline(y=0, color='r', linestyle='--')
    axes[0, 0].set_title('Residuales vs Tiempo')
    axes[0, 0].set_xlabel('Fecha')
    axes[0, 0].set_ylabel('Residuales')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Histograma de residuales
    axes[0, 1].hist(residuals_prophet, bins=10, edgecolor='black')
    axes[0, 1].set_title('Distribución de Residuales')
    axes[0, 1].set_xlabel('Residual')
    axes[0, 1].set_ylabel('Frecuencia')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(residuals_prophet, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title('Q-Q Plot')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Scatter plot: Predicho vs Observado
    axes[1, 1].scatter(forecast_test['yhat'], test_prophet['y'], alpha=0.6)
    axes[1, 1].plot([test_prophet['y'].min(), test_prophet['y'].max()], 
                     [test_prophet['y'].min(), test_prophet['y'].max()], 
                     'r--', lw=2)
    axes[1, 1].set_xlabel('Predicho')
    axes[1, 1].set_ylabel('Observado')
    axes[1, 1].set_title('Predicho vs Observado')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    residuals_filename = "prophet_residuals_analysis.png"
    plt.savefig(residuals_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(residuals_filename, artifact_path="plots")
    plt.close()
    
    # 8. Guardar forecast completo (train + test)
    forecast_complete_filename = "prophet_forecast_complete.csv"
    forecast.to_csv(forecast_complete_filename, index=False)
    mlflow.log_artifact(forecast_complete_filename, artifact_path="predictions")
    
    # 9. Guardar resumen del modelo como JSON
    model_summary = {
        'model_type': 'Prophet',
        'hyperparameters': {
            'changepoint_prior_scale': float(best_params_prophet['changepoint_prior_scale']),
            'seasonality_prior_scale': float(best_params_prophet['seasonality_prior_scale']),
            'holidays_prior_scale': float(best_params_prophet['holidays_prior_scale']),
            'seasonality_mode': best_params_prophet['seasonality_mode'],
            'changepoint_range': float(best_params_prophet['changepoint_range'])
        },
        'metrics': {
            'rmse': float(prophet_rmse),
            'mae': float(prophet_mae)
        },
        'training_period': '1960-1966',
        'test_period': '1967',
        'n_train': len(train_prophet),
        'n_test': len(test_prophet),
        'optimization': {
            'method': 'Optuna',
            'n_trials': 50,
            'best_trial_rmse': float(study_prophet.best_value)
        }
    }
    
    summary_filename = "prophet_model_summary.json"
    with open(summary_filename, 'w') as f:
        json.dump(model_summary, f, indent=2)
    mlflow.log_artifact(summary_filename, artifact_path="metadata")
    
    print("\n✓ Artifacts de Prophet registrados exitosamente en MLflow:")
    print("  - Modelo (pickle)")
    print("  - Predicciones (CSV)")
    print("  - Gráfico de pronóstico")
    print("  - Gráfico de componentes")
    print("  - Análisis de residuales")
    print("  - Forecast completo")
    print("  - Resumen del modelo (JSON)")

In [0]:
# Registrar comparación final de modelos
with mlflow.start_run(run_name="Model_Comparison_SARIMA_vs_Prophet"):
    
    # 1. Registrar métricas de ambos modelos
    mlflow.log_metrics({
        'sarima_rmse': final_rmse,
        'sarima_mae': final_mae,
        'prophet_rmse': prophet_rmse,
        'prophet_mae': prophet_mae,
        'rmse_difference': abs(final_rmse - prophet_rmse),
        'mae_difference': abs(final_mae - prophet_mae)
    })
    
    # 2. Crear y guardar tabla comparativa
    comparison_df = pd.DataFrame({
        'Modelo': ['SARIMA(0,2,2)(0,1,1,12)', 'Prophet Optimizado'],
        'RMSE': [final_rmse, prophet_rmse],
        'MAE': [final_mae, prophet_mae],
        'RMSE_Rank': [1.0, 2.0] if final_rmse < prophet_rmse else [2.0, 1.0],
        'MAE_Rank': [2.0, 1.0] if final_mae > prophet_mae else [1.0, 2.0]
    })
    
    comparison_filename = "model_comparison.csv"
    comparison_df.to_csv(comparison_filename, index=False)
    mlflow.log_artifact(comparison_filename, artifact_path="comparison")
    
    # 3. Crear gráfico comparativo lado a lado
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # SARIMA
    axes[0].plot(train_df.index, train_df['ventas'], label='Entrenamiento', 
                 color='blue', linewidth=2, alpha=0.5)
    axes[0].plot(test_df.index, test_df['ventas'], label='Observado', 
                 color='green', linewidth=2, marker='o', markersize=6)
    axes[0].plot(test_df.index, forecast_1967, label='SARIMA', 
                 color='red', linewidth=2, linestyle='--', marker='s', markersize=6)
    axes[0].set_title(f'SARIMA - RMSE: {final_rmse:.2f}, MAE: {final_mae:.2f}',
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Fecha')
    axes[0].set_ylabel('Ventas')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Prophet
    axes[1].plot(train_prophet['ds'], train_prophet['y'], label='Entrenamiento', 
                 color='blue', linewidth=2, alpha=0.5)
    axes[1].plot(test_prophet['ds'], test_prophet['y'], label='Observado', 
                 color='green', linewidth=2, marker='o', markersize=6)
    axes[1].plot(forecast_test['ds'], forecast_test['yhat'], label='Prophet', 
                 color='orange', linewidth=2, linestyle='--', marker='^', markersize=6)
    axes[1].set_title(f'Prophet - RMSE: {prophet_rmse:.2f}, MAE: {prophet_mae:.2f}',
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Fecha')
    axes[1].set_ylabel('Ventas')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    comparison_plot_filename = "models_side_by_side_comparison.png"
    plt.savefig(comparison_plot_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(comparison_plot_filename, artifact_path="comparison")
    plt.close()
    
    # 4. Crear gráfico de métricas comparativas
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    models = ['SARIMA', 'Prophet']
    rmse_values = [final_rmse, prophet_rmse]
    mae_values = [final_mae, prophet_mae]
    
    # RMSE comparison
    colors = ['green' if rmse_values[0] < rmse_values[1] else 'orange', 
              'green' if rmse_values[1] < rmse_values[0] else 'orange']
    axes[0].bar(models, rmse_values, color=colors, alpha=0.7, edgecolor='black')
    axes[0].set_title('Comparación RMSE', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('RMSE')
    axes[0].grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(rmse_values):
        axes[0].text(i, v + 20, f'{v:.2f}', ha='center', fontweight='bold')
    
    # MAE comparison
    colors = ['green' if mae_values[0] < mae_values[1] else 'orange', 
              'green' if mae_values[1] < mae_values[0] else 'orange']
    axes[1].bar(models, mae_values, color=colors, alpha=0.7, edgecolor='black')
    axes[1].set_title('Comparación MAE', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('MAE')
    axes[1].grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(mae_values):
        axes[1].text(i, v + 20, f'{v:.2f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    metrics_comparison_filename = "metrics_bar_comparison.png"
    plt.savefig(metrics_comparison_filename, dpi=300, bbox_inches='tight')
    mlflow.log_artifact(metrics_comparison_filename, artifact_path="comparison")
    plt.close()
    
    # 5. Guardar resumen de comparación como JSON
    comparison_summary = {
        'comparison_date': pd.Timestamp.now().isoformat(),
        'models_compared': ['SARIMA(0,2,2)(0,1,1,12)', 'Prophet Optimizado'],
        'metrics': {
            'SARIMA': {
                'rmse': float(final_rmse),
                'mae': float(final_mae)
            },
            'Prophet': {
                'rmse': float(prophet_rmse),
                'mae': float(prophet_mae)
            }
        },
        'winner': {
            'by_rmse': 'SARIMA' if final_rmse < prophet_rmse else 'Prophet',
            'by_mae': 'SARIMA' if final_mae < prophet_mae else 'Prophet'
        },
        'improvement': {
            'prophet_vs_baseline': {
                'baseline_rmse': 2793.27,
                'optimized_rmse': float(prophet_rmse),
                'improvement_percent': ((2793.27 - prophet_rmse) / 2793.27) * 100
            }
        }
    }
    
    comparison_summary_filename = "comparison_summary.json"
    with open(comparison_summary_filename, 'w') as f:
        json.dump(comparison_summary, f, indent=2)
    mlflow.log_artifact(comparison_summary_filename, artifact_path="comparison")
    
    print("\n" + "="*70)
    print("ARTIFACTS DE COMPARACIÓN REGISTRADOS EN MLFLOW")
    print("="*70)
    print("✓ Tabla comparativa (CSV)")
    print("✓ Gráficos comparativos lado a lado")
    print("✓ Gráficos de barras de métricas")
    print("✓ Resumen de comparación (JSON)")
    print("\n" + "="*70)
    print(f"Experimento: /Users/joel@ramirezai.com/prueba_det_tiempo")
    print("Todos los artifacts están disponibles en MLflow UI")
    print("="*70)

In [0]:
# RESUMEN COMPLETO DE ARTIFACTS REGISTRADOS EN MLFLOW
print("="*80)
print("📊 RESUMEN DE ARTIFACTS REGISTRADOS EN MLFLOW")
print("="*80)

print("\n🎯 EXPERIMENTO:")
print(f"   Nombre: /Users/joel@ramirezai.com/prueba_de_series_de_tiempo")
print(f"   Total de runs: 3 (SARIMA, Prophet, Comparación)")

print("\n" + "="*80)
print("1️⃣  SARIMA(0,2,2)(0,1,1,12) - Artifacts Registrados")
print("="*80)
print("\n📦 Modelos:")
print("   ├─ sarima_model.pkl          (Modelo entrenado serializado)")
print("\n📈 Predicciones:")
print("   ├─ sarima_predictions.csv    (Observado vs Predicho + errores)")
print("\n📊 Gráficos:")
print("   ├─ sarima_forecast_plot.png          (Pronóstico con intervalos de confianza)")
print("   ├─ sarima_residuals_analysis.png     (4 gráficos: residuales, histograma, Q-Q, ACF)")
print("\n📋 Metadata:")
print("   ├─ sarima_model_summary.json (Hiperparámetros, métricas, info del modelo)")
print("\n📏 Métricas:")
print(f"   ├─ RMSE: {final_rmse:.2f}")
print(f"   ├─ MAE:  {final_mae:.2f}")
print(f"   ├─ AIC:  {final_fitted.aic:.2f}")
print(f"   └─ BIC:  {final_fitted.bic:.2f}")

print("\n" + "="*80)
print("2️⃣  PROPHET OPTIMIZADO - Artifacts Registrados")
print("="*80)
print("\n📦 Modelos:")
print("   ├─ prophet_model.pkl         (Modelo entrenado serializado)")
print("\n📈 Predicciones:")
print("   ├─ prophet_predictions.csv   (Observado vs Predicho + intervalos + errores)")
print("   ├─ prophet_forecast_complete.csv (Forecast completo: train + test)")
print("\n📊 Gráficos:")
print("   ├─ prophet_forecast_plot.png         (Pronóstico con intervalos de confianza)")
print("   ├─ prophet_components.png            (Tendencia + Estacionalidad anual)")
print("   ├─ prophet_residuals_analysis.png    (4 gráficos: residuales, histograma, Q-Q, scatter)")
print("\n📋 Metadata:")
print("   ├─ prophet_model_summary.json (Hiperparámetros, métricas, info de optimización)")
print("\n📏 Métricas:")
print(f"   ├─ RMSE: {prophet_rmse:.2f}")
print(f"   └─ MAE:  {prophet_mae:.2f}")
print("\n🔧 Optimización con Optuna:")
print(f"   ├─ Trials: 50")
print(f"   └─ Mejora vs baseline: {((2793.27 - prophet_rmse) / 2793.27) * 100:.1f}%")

print("\n" + "="*80)
print("3️⃣  COMPARACIÓN DE MODELOS - Artifacts Registrados")
print("="*80)
print("\n📊 Gráficos:")
print("   ├─ models_side_by_side_comparison.png (Pronósticos lado a lado)")
print("   ├─ metrics_bar_comparison.png         (Comparación visual de RMSE y MAE)")
print("\n📋 Tablas y Metadata:")
print("   ├─ model_comparison.csv               (Tabla con todas las métricas y rankings)")
print("   ├─ comparison_summary.json            (Resumen completo de la comparación)")
print("\n📏 Resultados Finales:")
print(f"   ├─ Ganador por RMSE: {'SARIMA' if final_rmse < prophet_rmse else 'Prophet'} ({min(final_rmse, prophet_rmse):.2f})")
print(f"   ├─ Ganador por MAE:  {'SARIMA' if final_mae < prophet_mae else 'Prophet'} ({min(final_mae, prophet_mae):.2f})")
print(f"   └─ Diferencia RMSE:  {abs(final_rmse - prophet_rmse):.2f}")

print("\n" + "="*80)
print("📂 CÓMO ACCEDER A LOS ARTIFACTS")
print("="*80)
print("\n1. Abrir la UI de MLflow:")
print("   - Click en 'Experiments' en el panel izquierdo")
print("   - Buscar: /Users/joel@ramirezai.com/prueba_de_series_de_tiempo")
print("\n2. Ver los runs:")
print("   - SARIMA_Final_Model_with_Artifacts")
print("   - Prophet_Final_Model_with_Artifacts")
print("   - Model_Comparison_SARIMA_vs_Prophet")
print("\n3. Descargar artifacts:")
print("   - Click en cualquier run")
print("   - Ir a la pestaña 'Artifacts'")
print("   - Descargar modelos, gráficos, predicciones, etc.")
print("\n4. Reproducir modelos:")
print("   - Los pickles (.pkl) pueden cargarse con pickle.load()")
print("   - Las predicciones están en CSV para análisis posterior")
print("   - Los JSONs contienen toda la configuración del modelo")

print("\n" + "="*80)
print("✅ TODOS LOS ARTIFACTS REGISTRADOS EXITOSAMENTE")
print("="*80)
print("\n💡 Tip: Cada run contiene métricas, parámetros y artifacts listos para")
print("   producción, reproducibilidad y comparación de modelos.")
print("="*80)

In [0]:
%pip install prophet

In [0]:
dbutils.library.restartPython()

In [0]:
# Recargar todas las librerías necesarias
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from prophet import Prophet
import optuna
import warnings
warnings.filterwarnings('ignore')

# Recargar datos
serie_mensual_df = spark.read.table("joel.default.parcial_1_serie_mensual").toPandas()
serie_mensual_df['mes'] = pd.to_datetime(serie_mensual_df['mes'])
serie_mensual_df = serie_mensual_df.set_index('mes')

# Recrear train y test
train_df = serie_mensual_df[serie_mensual_df.index.year <= 1966]
test_df = serie_mensual_df[serie_mensual_df.index.year == 1967]

print("Datos recargados exitosamente")
print(f"Entrenamiento: {len(train_df)} meses | Prueba: {len(test_df)} meses")

In [0]:
# Prophet requiere columnas 'ds' (fecha) y 'y' (valor)
train_prophet = pd.DataFrame({
    'ds': train_df.index,
    'y': train_df['ventas'].values
})

test_prophet = pd.DataFrame({
    'ds': test_df.index,
    'y': test_df['ventas'].values
})

print("Datos preparados para Prophet:")
print(f"\nTrain shape: {train_prophet.shape}")
print(f"Test shape: {test_prophet.shape}")
print(f"\nPrimeras filas de entrenamiento:")
display(train_prophet.head())

In [0]:
import optuna
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# Configurar experimento MLflow
mlflow.set_experiment("/Users/joel@ramirezai.com/prueba_de_series_de_tiempo")

def objective_prophet(trial):
    """Función objetivo para Optuna que optimiza hiperparámetros de Prophet"""
    
    # Sugerir hiperparámetros para Prophet
    changepoint_prior_scale = trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True)
    seasonality_prior_scale = trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True)
    holidays_prior_scale = trial.suggest_float('holidays_prior_scale', 0.01, 10.0, log=True)
    seasonality_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
    changepoint_range = trial.suggest_float('changepoint_range', 0.8, 0.95)
    
    try:
        with mlflow.start_run(nested=True):
            # Registrar hiperparámetros
            mlflow.log_params({
                'changepoint_prior_scale': changepoint_prior_scale,
                'seasonality_prior_scale': seasonality_prior_scale,
                'holidays_prior_scale': holidays_prior_scale,
                'seasonality_mode': seasonality_mode,
                'changepoint_range': changepoint_range
            })
            
            # Crear y entrenar modelo Prophet
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=changepoint_prior_scale,
                seasonality_prior_scale=seasonality_prior_scale,
                holidays_prior_scale=holidays_prior_scale,
                seasonality_mode=seasonality_mode,
                changepoint_range=changepoint_range
            )
            
            model.fit(train_prophet)
            
            # Crear DataFrame de fechas futuras y hacer predicciones
            future = model.make_future_dataframe(periods=12, freq='MS')
            forecast = model.predict(future)
            
            # Extraer predicciones para conjunto de prueba
            forecast_test = forecast.tail(12)['yhat'].values
            
            # Calcular métricas
            rmse = np.sqrt(mean_squared_error(test_prophet['y'], forecast_test))
            mae = mean_absolute_error(test_prophet['y'], forecast_test)
            
            # Registrar métricas
            mlflow.log_metrics({
                'rmse': rmse,
                'mae': mae
            })
            
            return rmse  # Optuna minimizará el RMSE
            
    except Exception as e:
        # Si el modelo falla, retornar un valor muy alto
        return float('inf')

print("Iniciando optimización de hiperparámetros de Prophet con Optuna...")
print("Este proceso puede tomar varios minutos.\n")

In [0]:
# Crear y ejecutar el estudio de Optuna para Prophet
with mlflow.start_run(run_name="Prophet_Optuna_Optimization"):
    study_prophet = optuna.create_study(direction='minimize', study_name='prophet_optimization')
    study_prophet.optimize(objective_prophet, n_trials=50, show_progress_bar=True)
    
    # Registrar los mejores hiperparámetros
    best_params_prophet = study_prophet.best_params
    mlflow.log_params({
        f'best_{k}': v for k, v in best_params_prophet.items()
    })
    mlflow.log_metric('best_rmse', study_prophet.best_value)
    
    print("\n" + "="*60)
    print("OPTIMIZACIÓN PROPHET COMPLETADA")
    print("="*60)
    print(f"\nMejores hiperparámetros encontrados:")
    for key, value in best_params_prophet.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.6f}")
        else:
            print(f"  {key}: {value}")
    print(f"\nMejor RMSE: {study_prophet.best_value:.2f}")

In [0]:
# Entrenar el modelo final con los mejores hiperparámetros
print("\nEntrenando modelo Prophet final con los mejores parámetros...")

model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=best_params_prophet['changepoint_prior_scale'],
    seasonality_prior_scale=best_params_prophet['seasonality_prior_scale'],
    holidays_prior_scale=best_params_prophet['holidays_prior_scale'],
    seasonality_mode=best_params_prophet['seasonality_mode'],
    changepoint_range=best_params_prophet['changepoint_range']
)

model_prophet.fit(train_prophet)

# Crear DataFrame de fechas futuras (12 meses de 1967)
future = model_prophet.make_future_dataframe(periods=12, freq='MS')

# Hacer predicciones
forecast = model_prophet.predict(future)

# Extraer solo las predicciones para el conjunto de prueba
forecast_test = forecast.tail(12)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

# Calcular métricas finales
prophet_rmse = np.sqrt(mean_squared_error(test_prophet['y'], forecast_test['yhat']))
prophet_mae = mean_absolute_error(test_prophet['y'], forecast_test['yhat'])

print("\n" + "="*60)
print("MODELO PROPHET - MÉTRICAS EN CONJUNTO DE PRUEBA (1967)")
print("="*60)
print(f"RMSE: {prophet_rmse:.2f}")
print(f"MAE:  {prophet_mae:.2f}")
print(f"\nComparación con SARIMA:")
print(f"SARIMA RMSE: 1523.71")
print(f"Prophet RMSE: {prophet_rmse:.2f}")
print(f"Diferencia: {prophet_rmse - 1523.71:.2f}")

In [0]:
# Crear visualización comparativa
fig, ax = plt.subplots(figsize=(14, 6))

# Graficar datos de entrenamiento
ax.plot(train_prophet['ds'], train_prophet['y'], 
        label='Entrenamiento (1960-1966)', 
        color='blue', linewidth=2, alpha=0.7)

# Graficar datos observados de 1967
ax.plot(test_prophet['ds'], test_prophet['y'], 
        label='Observado (1967)', 
        color='green', linewidth=2, marker='o', markersize=6)

# Graficar pronóstico de Prophet
ax.plot(forecast_test['ds'], forecast_test['yhat'], 
        label='Pronosticado Prophet (1967)', 
        color='red', linewidth=2, linestyle='--', marker='s', markersize=6)

# Agregar intervalo de confianza de Prophet
ax.fill_between(forecast_test['ds'], 
                 forecast_test['yhat_lower'], 
                 forecast_test['yhat_upper'],
                 alpha=0.2, color='red', label='Intervalo de confianza 80%')

ax.set_title('Pronóstico Prophet - Observado vs Pronosticado',
            fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Ventas', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()

# Agregar texto con métricas
textstr = f'RMSE: {prophet_rmse:.2f}\nMAE: {prophet_mae:.2f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

In [0]:
# Visualizar los componentes del modelo Prophet
fig = model_prophet.plot_components(forecast)
plt.tight_layout()
plt.show()

In [0]:
# Crear tabla comparativa de modelos
comparacion = pd.DataFrame({
    'Modelo': ['SARIMA(0,2,2)(0,1,1,12)', 'Prophet'],
    'RMSE': [1523.71, prophet_rmse],
    'MAE': [1393.58, prophet_mae]
})

comparacion['RMSE_Rank'] = comparacion['RMSE'].rank()
comparacion['MAE_Rank'] = comparacion['MAE'].rank()

print("\n" + "="*70)
print("COMPARACIÓN DE MODELOS - CONJUNTO DE PRUEBA (1967)")
print("="*70)
print(comparacion.to_string(index=False))
print("\n" + "="*70)

# Determinar el mejor modelo
mejor_modelo_rmse = comparacion.loc[comparacion['RMSE'].idxmin(), 'Modelo']
mejor_modelo_mae = comparacion.loc[comparacion['MAE'].idxmin(), 'Modelo']

print(f"\nMejor modelo por RMSE: {mejor_modelo_rmse}")
print(f"Mejor modelo por MAE: {mejor_modelo_mae}")

# Calcular mejora porcentual
if comparacion.iloc[0]['RMSE'] < comparacion.iloc[1]['RMSE']:
    mejora_pct = ((comparacion.iloc[1]['RMSE'] - comparacion.iloc[0]['RMSE']) / comparacion.iloc[1]['RMSE']) * 100
    print(f"\nSARIMA es {mejora_pct:.2f}% mejor que Prophet en RMSE")
else:
    mejora_pct = ((comparacion.iloc[0]['RMSE'] - comparacion.iloc[1]['RMSE']) / comparacion.iloc[0]['RMSE']) * 100
    print(f"\nProphet es {mejora_pct:.2f}% mejor que SARIMA en RMSE")

# Parte 2 Prueba

a) Explique con sus palabras qué hace la regularización (Ridge y Lasso) sobre los coeficientes de una regresión y por qué ayuda a controlar el sobreajuste.

Ridge penaliza la suma de los cuadrados de los coeficientes. Ridge encoge los coeficientes de forma proporcional, pero casi nunca los lleva a 0, pero ayuda a controlar el ruido

Por otro lado, Lasso penaliza la suma del valor absoluto de los coeficientes. Lo que le permite llevar a 0 algunos de los coeficientes que no aportan. Por lo que ayuda a seleccionar las variables 




b) Mencione una diferencia práctica entre Ridge y Lasso y relaciónela con el compromiso
sesgo–varianza.

Cuando se cree tener variables que no aportan al modelo, podemos usar lasso para eliminarlas y simplificar el modelo y su interpretabilidad y Ridge lo usamos cuando creemos que todas las variables aportan algo modelo. 

Respecto al sesgo-varianza
 - Ridge reduce la varianza de forma continua y suave
 - Lasso reduce la varianza de forma discontinua. lo que le permite apagar ciertos coeficientes por completo


R

Considere la matriz de transición de dos estados (0 = mes de temporada baja, 1 = mes
de temporada alta):

Matriz de transición:

|        | 0 (baja) | 1 (alta) |
|--------|----------|----------|
| **0**  |   0.8    |   0.2    |
| **1**  |   0.5    |   0.5    |

- a) Interprete en palabras qué dice la primera fila.
- b) ¿Qué representa la distribución estacionaria de esta cadena? Plantee la ecuación
πP = π (no hace falta resolverla a mano si la razona correctamente).
- c) En una frase, conecte esta idea con PageRank o con los modelos de lenguaje.

a) **Primera fila**  
Si este mes es temporada baja: 80% de chance de seguir en baja, 20% de pasar a alta.

b) **Distribución estacionaria**  
Es el reparto de largo plazo: qué % de meses termina siendo baja y qué % alta, si pasa mucho tiempo.  
La ecuación **π · P = π** dice: "ese reparto ya no cambia si le aplicas un mes más."
Como la baja es más "pegajosa" (80% vs 50%), a largo plazo hay más meses en baja que en alta:  
**π ≈ [0.71, 0.29]**

c) **Conexión**  
PageRank hace lo mismo pero con páginas web: calcula en qué páginas terminaría "a la larga" alguien que hace clics al azar, para saber cuáles son más importantes.


Observe el siguiente fragmento:

import numpy as np

rng = np.random. default_rng (2026)

n = 100000

x = rng.uniform (-1, 1, n)

y = rng.uniform (-1, 1, n)

dentro = (x**2 + y**2) <= 1

print(4 * dentro.mean ())

- a) ¿Qué estima este código y por qué funciona? (¿es Monte Carlo o MCMC?)
- b) Explique en dos frases la diferencia entre simulación de Monte Carlo y MCMC, y para
qué sirve cada una.

In [0]:
import numpy as np
rng = np.random. default_rng (2026)
n = 100000
x = rng.uniform (-1, 1, n)
y = rng.uniform (-1, 1, n)
dentro = (x**2 + y**2) <= 1
print(4 * dentro.mean ())

a) ¿Qué estima?

Estima π (pi).
Tira 100,000 puntos al azar dentro de un cuadrado. Cuenta cuántos caen dentro de un círculo inscrito en ese cuadrado. Esa proporción, multiplicada por 4, se acerca a π.
Es Monte Carlo, no MCMC — porque cada punto se genera solo, sin depender de los anteriores.

b) Monte Carlo vs MCMC

Monte Carlo: tiras puntos al azar, todos independientes entre sí. Sirve cuando puedes generar puntos directamente (como aquí).
MCMC: cada punto depende del anterior, como una cadena. Sirve cuando la distribución es tan complicada que no puedes generar puntos directamente, solo puedes ir dando pasos hacia ella poco a poco.

# Parte 3 Ejercicio a mano 
Resulado Se encuentra en el zip